# AI Learning Assistant — Model Training (Epic 2)

Run this notebook on **Kaggle** with **GPU** enabled (Settings → Accelerator → GPU T4 x2).

Covers:
- Story 1: Kaggle Setup (GPU + Dependencies + Data Loading)
- Story 2: Data Preprocessing & Tokenization
- Story 3: BiLSTM Model Training
- Story 4: Domain-Adaptive Fine-Tuning
- Story 5: BERT Model Fine-Tuning
- Story 6: Model Export & Local Integration

**Labels (5-class):** Bored, Confident, Confused, Curious, Frustrated

**Before running:** upload a labeled CSV dataset with columns `text,label` as a
Kaggle Dataset and attach it to this notebook (Add Data), or point `DATA_PATH`
at your own source.

## Story 1: Setup — GPU check + dependencies

In [ ]:
!pip install -q transformers torch scikit-learn nltk

import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import pandas as pd

# Point this at your attached Kaggle dataset, e.g. '/kaggle/input/your-dataset/data.csv'
DATA_PATH = '/kaggle/input/learning-emotions/data.csv'
LABELS = ['Bored', 'Confident', 'Confused', 'Curious', 'Frustrated']

df = pd.read_csv(DATA_PATH)
assert set(['text', 'label']).issubset(df.columns), "CSV must have 'text' and 'label' columns"
df = df[df['label'].isin(LABELS)].reset_index(drop=True)
print(df.shape)
df.head()

## Story 2: Data Preprocessing & Tokenization

In [ ]:
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def clean_text(t):
    t = str(t).lower().strip()
    t = re.sub(r"[^a-z0-9\s']", ' ', t)
    t = re.sub(r'\s+', ' ', t)
    return t

df['clean_text'] = df['text'].apply(clean_text)

label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['label'])

train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label_id'], random_state=42)
print('Train:', len(train_df), 'Val:', len(val_df))

## Story 3: BiLSTM Model Training

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

MAX_LEN = 40
VOCAB_SIZE = 15000
EMBED_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = len(LABELS)

# Build vocabulary
counter = Counter()
for t in train_df['clean_text']:
    counter.update(t.split())
vocab = {w: i + 2 for i, (w, _) in enumerate(counter.most_common(VOCAB_SIZE - 2))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

def encode(text, max_len=MAX_LEN):
    ids = [vocab.get(w, 1) for w in text.split()][:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

class EmotionDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['clean_text'].tolist()
        self.labels = frame['label_id'].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_loader = DataLoader(EmotionDataset(train_df), batch_size=32, shuffle=True)
val_loader = DataLoader(EmotionDataset(val_df), batch_size=32)

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        pooled = out.mean(dim=1)
        return self.fc(self.dropout(pooled))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bilstm = BiLSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(bilstm.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 8
for epoch in range(EPOCHS):
    bilstm.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = bilstm(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} - loss: {total_loss/len(train_loader):.4f}')

## Story 4: Domain-Adaptive Fine-Tuning
Fine-tune on a smaller, education-specific subset (e.g. tutoring transcripts, forum posts) to adapt away from generic sentiment data.

In [ ]:
# If you have a smaller education-domain dataset, load it here and
# continue training `bilstm` for a few more epochs at a lower LR.
# DOMAIN_DATA_PATH = '/kaggle/input/education-domain/domain.csv'
# domain_df = pd.read_csv(DOMAIN_DATA_PATH)
# domain_df['clean_text'] = domain_df['text'].apply(clean_text)
# domain_df['label_id'] = label_encoder.transform(domain_df['label'])
# domain_loader = DataLoader(EmotionDataset(domain_df), batch_size=16, shuffle=True)
# for g in optimizer.param_groups: g['lr'] = 2e-4
# for epoch in range(3):
#     bilstm.train()
#     for x, y in domain_loader:
#         x, y = x.to(device), y.to(device)
#         optimizer.zero_grad()
#         loss = criterion(bilstm(x), y)
#         loss.backward(); optimizer.step()
print('Fill in domain-specific dataset path above to run adaptive fine-tuning.')

## Story 5: BERT Model Fine-Tuning (with class weighting)

In [ ]:
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

class BertDataset(Dataset):
    def __init__(self, frame):
        enc = tokenizer(list(frame['clean_text']), truncation=True, padding=True, max_length=64)
        self.enc = enc
        self.labels = frame['label_id'].tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_ds = BertDataset(train_df)
val_ds = BertDataset(val_df)

class_weights = compute_class_weight('balanced', classes=np.unique(df['label_id']), y=df['label_id'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=NUM_CLASSES
).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

args = TrainingArguments(
    output_dir='/kaggle/working/bert_out',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(model=bert_model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

## Story 6: Model Export & Local Integration
Download these folders from Kaggle's Output tab and place them exactly here in your local project:
- `models/bltsm/` ← BiLSTM export
- `models/bert_emotion_model_final/` ← BERT export

In [ ]:
import pickle, os

os.makedirs('/kaggle/working/bltsm', exist_ok=True)
torch.save(bilstm.state_dict(), '/kaggle/working/bltsm/bilstm_model.pt')
with open('/kaggle/working/bltsm/tokenizer.pkl', 'wb') as f:
    pickle.dump(vocab, f)
with open('/kaggle/working/bltsm/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

os.makedirs('/kaggle/working/bert_emotion_model_final', exist_ok=True)
trainer.save_model('/kaggle/working/bert_emotion_model_final')
tokenizer.save_pretrained('/kaggle/working/bert_emotion_model_final')

print('Export complete. Download the /kaggle/working folders and copy them into your local models/ directory.')